# DLGenAI Project 26T3 - Milestone 4
## Optimization Dynamics, Multi-Level Class Imbalance & Metric Governance

### Instructions & Setup
- **W&B Project**: `24F2004781-t32026`
- **W&B Entity**: `dineshwar-developer-indian-institute-of-technology`
- **Subset**: 1,200 unique ASCII-sorted image IDs (4,800 rows)
- **Split**: 80/20 Stratified on defect combination string (rare < 3 -> `rare_other`) -> 960 train / 240 val images
- **Canvas**: Native 128x800 canvas (Fortran order, 1-based, run clipping at 102,400)
- **Fixed Training Budget**: native 128x800 input, AdamW (lr 3e-4), batch size 16, AMP on, seed 42, decision threshold 0.5, no minimum-area filtering.

In [ ]:
# Common Environment & Path Setup
import os
import sys
import time
import gc
import ctypes
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Apply glibc memory optimizations if running on Linux / Kaggle
try:
    ctypes.CDLL('libc.so.6').mallopt(ctypes.c_int(-100), ctypes.c_int(0))
    ctypes.CDLL('libc.so.6').mallopt(ctypes.c_int(-101), ctypes.c_int(1))
except Exception:
    pass

cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

# W&B Configuration
try:
    import wandb
except ImportError:
    wandb = None

class CFG:
    SEED = 42
    WANDB_PROJECT = '24F2004781-t32026'
    WANDB_ENTITY = 'dineshwar-developer-indian-institute-of-technology'
    CANVAS_HEIGHT = 128
    CANVAS_WIDTH = 800
    CANVAS_SIZE = 128 * 800  # 102,400
    DEFECT_CLASSES = ['defect_a', 'defect_b', 'defect_c', 'defect_d']
    LR = 3e-4
    BATCH_SIZE = 16
    THRESHOLD = 0.5

# Automatic Local vs. Kaggle Path Resolution
def resolve_paths():
    candidates = [
        ('/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv', '/kaggle/input/competitions/industrial-surface-defect-detection/public/train'),
        ('/kaggle/input/industrial-surface-defect-detection/public/train.csv', '/kaggle/input/industrial-surface-defect-detection/public/train'),
        ('/kaggle/input/industrial-surface-defect-detection/train.csv', '/kaggle/input/industrial-surface-defect-detection/train'),
        ('data/train.csv', 'data/train'),
        ('../data/train.csv', '../data/train'),
    ]
    for csv_p, img_p in candidates:
        if os.path.exists(csv_p):
            return csv_p, img_p
    return 'data/train.csv', 'data/train'

DATA_CSV, IMAGE_DIR = resolve_paths()
print(f'Resolved DATA_CSV: {DATA_CSV}')
print(f'Resolved IMAGE_DIR: {IMAGE_DIR}')


In [ ]:
# Common Setup: Working Subset, Native RLE Decoding & Stratified Split
def rle_to_mask(rle_string, height=128, width=800):
    """Decodes RLE mask on native 128x800 canvas (Fortran order, 1-based, run clipping at 102,400)."""
    if pd.isna(rle_string) or not str(rle_string).strip():
        return np.zeros((height, width), dtype=np.uint8)
    canvas_size = height * width
    mask_flat = np.zeros(canvas_size, dtype=np.uint8)
    runs = np.fromstring(str(rle_string), dtype=np.int32, sep=' ')
    starts = runs[0::2] - 1  # Convert 1-based to 0-based
    lengths = runs[1::2]
    for s, l in zip(starts, lengths):
        e = min(s + l, canvas_size)  # Clip runs exceeding canvas
        if s < canvas_size:
            mask_flat[s:e] = 1
    return mask_flat.reshape((height, width), order='F')

# 1. Load train.csv
df_all = pd.read_csv(DATA_CSV)

# 2. Sort unique image IDs alphabetically (ASCII order) and take first 1,200
unique_1200_ids = sorted(df_all['image_id'].unique())[:1200]
df_subset = df_all[df_all['image_id'].isin(unique_1200_ids)].copy().reset_index(drop=True)
print(f'Working subset: {len(unique_1200_ids)} unique image IDs, {len(df_subset)} image-class pairs.')

# 3. Build each image combination string
def get_combo(group):
    defects = group[group['mask_rle'].notna() & (group['mask_rle'].astype(str).str.strip() != '')]['defect_type'].tolist()
    if not defects:
        return 'clean'
    return '+'.join(sorted(defects))

combos = df_subset.groupby('image_id').apply(get_combo)
combo_counts = combos.value_counts()

# Merge combinations with fewer than 3 images into rare_other
rare_combos = combo_counts[combo_counts < 3].index
strat_labels = combos.apply(lambda x: 'rare_other' if x in rare_combos else x)

# 4. 80/20 Stratified Split with random_state=42
train_image_ids, val_image_ids = train_test_split(
    list(strat_labels.index),
    test_size=0.20,
    random_state=42,
    stratify=strat_labels.tolist()
)
train_image_ids = set(train_image_ids)
val_image_ids = set(val_image_ids)
print(f'Split complete: {len(train_image_ids)} train images / {len(val_image_ids)} validation images.')


---
## Question 1: Image-Level and Class-Level Imbalance

Use the 1,200-image subset (the full subset, not the split).

### Question:
(a) How many images are completely clean?
(b) How many images are positive for each of `defect_a`, `defect_b`, `defect_c`, `defect_d`?
(c) How many unique combination strings exist (before any merging)?
(d) What are the sizes of the largest and the smallest combination group?

**Answer format:** `a; b_a, b_b, b_c, b_d; c; d_largest, d_smallest` (all integers).


In [ ]:
# ==============================================================================
# Question 1: Image-Level and Class-Level Imbalance
# ==============================================================================
# Check positive status for each row
df_subset['is_positive'] = df_subset['mask_rle'].notna() & (df_subset['mask_rle'].astype(str).str.strip() != '')

# (a) Completely clean images
pos_per_image = df_subset.groupby('image_id')['is_positive'].sum()
ans_a = int((pos_per_image == 0).sum())

# (b) Positive images for each defect type
b_counts = {}
for defect in CFG.DEFECT_CLASSES:
    cnt = df_subset[(df_subset['defect_type'] == defect) & df_subset['is_positive']]['image_id'].nunique()
    b_counts[defect] = int(cnt)
ans_b_a = b_counts['defect_a']
ans_b_b = b_counts['defect_b']
ans_b_c = b_counts['defect_c']
ans_b_d = b_counts['defect_d']

# (c) Unique combination strings (before merging)
ans_c = int(combos.nunique())

# (d) Largest and smallest combination group sizes
ans_d_largest = int(combo_counts.max())
ans_d_smallest = int(combo_counts.min())

print(f'(a) Completely clean images: {ans_a}')
print(f'(b) Positive counts per defect: a={ans_b_a}, b={ans_b_b}, c={ans_b_c}, d={ans_b_d}')
print(f'(c) Unique combinations: {ans_c}')
print(f'(d) Largest group size: {ans_d_largest}, Smallest group size: {ans_d_smallest}')

# Final Formatted Answer
q1_answer = f'{ans_a}; {ans_b_a}, {ans_b_b}, {ans_b_c}, {ans_b_d}; {ans_c}; {ans_d_largest}, {ans_d_smallest}'
print('\n' + '='*40)
print(f'Question 1 Final Answer: {q1_answer}')
print('='*40)


---
## Question 2: Pixel-Level Imbalance and pos_weight

Use the same 1,200-image subset and decode all `defect_b` masks.

### Question:
(a) What percentage of all `defect_b` mask pixels (1,200 × 102,400) is positive? Give 4 decimals.
(b) What is the mean `defect_b` area (in pixels) among `defect_b`-positive images? Give 1 decimal.
(c) What is the ratio of negative to positive pixels (a candidate `pos_weight` for BCE)? Round to the nearest integer.

**Answer format:** `a (%, 4 decimals), b (1 decimal), c (integer)`.


In [ ]:
# ==============================================================================
# Question 2: Pixel-Level Imbalance and pos_weight
# ==============================================================================
b_rows = df_subset[df_subset['defect_type'] == 'defect_b'].copy()

# Decode all defect_b masks
b_areas = []
for _, row in b_rows.iterrows():
    mask = rle_to_mask(row['mask_rle'], height=CFG.CANVAS_HEIGHT, width=CFG.CANVAS_WIDTH)
    b_areas.append(int(mask.sum()))

b_areas = np.array(b_areas)
pos_b_areas = b_areas[b_areas > 0]

total_pixels = len(b_rows) * CFG.CANVAS_SIZE  # 1,200 * 102,400 = 122,880,000
total_pos_pixels = int(b_areas.sum())
total_neg_pixels = total_pixels - total_pos_pixels

# (a) Percentage of positive pixels (4 decimals)
pct_pos = (total_pos_pixels / total_pixels) * 100
ans_q2_a = f"{pct_pos:.4f}%"
ans_q2_a_num = f"{pct_pos:.4f}"

# (b) Mean defect_b area among positive images (1 decimal)
mean_b_area = float(pos_b_areas.mean())
ans_q2_b = f"{mean_b_area:.1f}"

# (c) Ratio of negative to positive pixels (nearest integer)
ratio_neg_pos = total_neg_pixels / total_pos_pixels
ans_q2_c = int(round(ratio_neg_pos))

print(f"Total pixels: {total_pixels:,}")
print(f"Total positive defect_b pixels: {total_pos_pixels:,}")
print(f"Total negative defect_b pixels: {total_neg_pixels:,}")
print(f"Number of defect_b positive images: {len(pos_b_areas)}")
print(f"(a) Percentage positive: {ans_q2_a} ({ans_q2_a_num})")
print(f"(b) Mean positive area: {ans_q2_b} px")
print(f"(c) Ratio negative/positive (pos_weight): {ans_q2_c}")

# Final Formatted Answer
q2_answer_with_pct = f"{ans_q2_a}, {ans_q2_b}, {ans_q2_c}"
q2_answer_clean = f"{ans_q2_a_num}, {ans_q2_b}, {ans_q2_c}"
print("\n" + "="*40)
print(f"Question 2 Final Answer: {q2_answer_with_pct}")
print(f"Question 2 (clean format without %): {q2_answer_clean}")
print("="*40)


---
## Question 3: Bounded Square-Root Sampling

Use the combination groups from Question 1 (all 1,200 images, no merging). Each image in group $g$ gets the weight
$$w_g = \min\left(\sqrt{\frac{n_{\text{clean}}}{n_g}}, 5\right),$$
where $n_g$ is the size of group $g$ and $n_{\text{clean}}$ is the size of the `clean` group. Draws are made with replacement, with probability proportional to weight, and one epoch is 1,200 draws.

### Question:
(a) What fraction of draws is expected to be clean, under this sampler and under natural uniform sampling?
(b) What is the expected number of draws per epoch of `defect_b`-positive images (any group containing `defect_b`), under this sampler and under natural sampling?
(c) By what factor does the sampler enrich `defect_b`-positive images?
(d) How many combination groups hit the cap of 5?

**Answer format:** `a_sampler %, a_natural % (2 decimals); b_sampler, b_natural (2 decimals); c (2 decimals, x); d (integer)`.


In [ ]:
# ==============================================================================
# Question 3: Bounded Square-Root Sampling
# ==============================================================================
# 1. Calculate weights for all 15 combination groups
n_clean = combo_counts['clean']
group_records = []

for g, n_g in combo_counts.items():
    raw_w = np.sqrt(n_clean / n_g)
    w_g = min(raw_w, 5.0)
    capped = bool(raw_w >= 5.0)
    has_b = ('defect_b' in g)
    total_w_g = n_g * w_g
    group_records.append({
        'group': g,
        'n_g': n_g,
        'raw_w': raw_w,
        'w_g': w_g,
        'capped': capped,
        'has_b': has_b,
        'total_w_g': total_w_g
    })

df_groups = pd.DataFrame(group_records)
total_weight = df_groups['total_w_g'].sum()

# (a) Fraction clean under sampler vs natural uniform sampling (in %, 2 decimals)
w_clean_total = df_groups[df_groups['group'] == 'clean']['total_w_g'].values[0]
frac_clean_sampler = (w_clean_total / total_weight) * 100
frac_clean_natural = (n_clean / 1200.0) * 100
ans_q3_a = f"{frac_clean_sampler:.2f}%, {frac_clean_natural:.2f}%"

# (b) Expected defect_b draws per epoch (1,200 draws) under sampler vs natural (2 decimals)
b_groups = df_groups[df_groups['has_b']]
w_b_total = b_groups['total_w_g'].sum()
n_b_total = b_groups['n_g'].sum()

expected_b_sampler = 1200.0 * (w_b_total / total_weight)
expected_b_natural = 1200.0 * (n_b_total / 1200.0)  # exactly n_b_total = 35
ans_q3_b = f"{expected_b_sampler:.2f}, {expected_b_natural:.2f}"

# (c) Defect_b enrichment factor (2 decimals, x)
enrichment_factor = expected_b_sampler / expected_b_natural
ans_q3_c = f"{enrichment_factor:.2f}x"

# (d) Number of combination groups hitting the cap of 5 (integer)
num_capped = int(df_groups['capped'].sum())
ans_q3_d = num_capped

print(f"Total weight of all 1,200 images: {total_weight:.4f}")
print(f"(a) Expected clean fraction: {ans_q3_a}")
print(f"(b) Expected defect_b draws per epoch: {ans_q3_b}")
print(f"(c) Defect_b enrichment factor: {ans_q3_c}")
print(f"(d) Groups hitting cap of 5: {ans_q3_d}")

# Final Formatted Answer
q3_answer = f"{ans_q3_a}; {ans_q3_b}; {ans_q3_c}; {ans_q3_d}"
print("\n" + "="*40)
print(f"Question 3 Final Answer: {q3_answer}")
print("="*40)


---
## Question 4: Warmup + Cosine Scheduler and the Stepping-Cadence Trap

Your training split has 960 images and batch size 16, so there are **60 optimizer steps per epoch**. You train for **8 epochs (480 steps)** with base $\text{LR} = 3 \times 10^{-4}$. A `LambdaLR` uses this multiplier:
$$m(t) = \begin{cases} t / 60, & t < 60 \quad (\text{linear warmup}) \\ 0.5 \left(1 + \cos\left(\pi \frac{t - 60}{420}\right)\right), & t \ge 60. \end{cases}$$

### Question:
(a) What is the learning rate at steps 15, 270, and 400?
(b) A student writes this schedule in units of steps but calls `scheduler.step()` once per epoch. What LR is the optimizer using after epoch 8?
(c) Briefly (one sentence): what symptom would you see in the training curves with this bug?

**Answer format:** `lr_15, lr_270, lr_400 (scientific notation, 3 significant figures); lr_bug (scientific notation, 3 significant figures); c (one sentence)`.


In [ ]:
# ==============================================================================
# Question 4: Warmup + Cosine Scheduler and the Stepping-Cadence Trap
# ==============================================================================
base_lr = 3e-4

def m(t):
    if t < 60:
        return t / 60.0
    else:
        return 0.5 * (1.0 + np.cos(np.pi * (t - 60.0) / 420.0))

# (a) Learning rate at steps 15, 270, and 400
lr_15 = base_lr * m(15)
lr_270 = base_lr * m(270)
lr_400 = base_lr * m(400)

ans_q4_a = f"{lr_15:.2e}, {lr_270:.2e}, {lr_400:.2e}"

# (b) LR after epoch 8 with epoch-level stepping cadence (t = 8)
t_bug = 8
lr_bug = base_lr * m(t_bug)
ans_q4_b = f"{lr_bug:.2e}"

# (c) Symptom observed in the training curve (one sentence)
ans_q4_c = "The training loss decreases extremely slowly and underfits because the learning rate remains trapped in the linear warmup phase throughout all 8 epochs, never reaching the peak learning rate."

print(f"(a) LRs at steps 15, 270, 400: {ans_q4_a}")
print(f"(b) Buggy LR after epoch 8: {ans_q4_b}")
print(f"(c) Curve symptom: {ans_q4_c}")

# Final Formatted Answer
q4_answer = f"{ans_q4_a}; {ans_q4_b}; {ans_q4_c}"
print("\n" + "="*40)
print(f"Question 4 Final Answer: {q4_answer}")
print("="*40)


---
## Question 5: Early Stopping Trace

You monitor the official validation score (`mode = max`) with `patience = 3` and `min_delta = 0.002`. An epoch counts as an improvement only if `score > best + min_delta`. Otherwise the patience counter increases by 1 (and resets to 0 on an improvement). The best checkpoint is saved only on improvement.

| Epoch | Val score |
| :---: | :---: |
| 1 | 0.512 |
| 2 | 0.601 |
| 3 | 0.648 |
| 4 | 0.661 |
| 5 | 0.664 |
| 6 | 0.668 |
| 7 | 0.671 |
| 8 | 0.670 |
| 9 | 0.672 |
| 10 | 0.669 |
| 11 | 0.673 |
| 12 | 0.665 |

### Question:
(a) At which epoch does training stop, which epoch's checkpoint is restored, and what is its score?
(b) With `min_delta = 0`, does training stop within these 12 epochs? Which epoch holds the best checkpoint at epoch 12?

**Answer format:** `e_stop, e_restored, s_restored; yes/no, e_best`.


In [ ]:
# ==============================================================================
# Question 5: Early Stopping Trace
# ==============================================================================
history_data = [
    (1, 0.512), (2, 0.601), (3, 0.648), (4, 0.661),
    (5, 0.664), (6, 0.668), (7, 0.671), (8, 0.670),
    (9, 0.672), (10, 0.669), (11, 0.673), (12, 0.665)
]

def run_early_stopping(history, patience=3, min_delta=0.002):
    best_score = -float('inf')
    best_epoch = None
    counter = 0
    stopped_epoch = None
    
    for ep, sc in history:
        if sc > best_score + min_delta:
            best_score = sc
            best_epoch = ep
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                stopped_epoch = ep
                break
    return stopped_epoch, best_epoch, best_score

# (a) Standard run with min_delta = 0.002, patience = 3
e_stop, e_restored, s_restored = run_early_stopping(history_data, patience=3, min_delta=0.002)
ans_q5_a = f"{e_stop}, {e_restored}, {s_restored:.3f}"

# (b) Run with min_delta = 0.0, patience = 3
e_stop_b, e_best_b, s_best_b = run_early_stopping(history_data, patience=3, min_delta=0.0)
stopped_within_12 = "yes" if e_stop_b is not None else "no"
ans_q5_b = f"{stopped_within_12}, {e_best_b}"

print(f"(a) Training stops at: Epoch {e_stop}, Checkpoint restored: Epoch {e_restored} (Score: {s_restored:.3f})")
print(f"(b) Stops within 12 epochs (min_delta=0)? {stopped_within_12}. Best checkpoint at epoch 12: Epoch {e_best_b} (Score: {s_best_b:.3f})")

# Final Formatted Answer
q5_answer = f"{ans_q5_a}; {ans_q5_b}"
print("\n" + "="*40)
print(f"Question 5 Final Answer: {q5_answer}")
print("="*40)


---
## Question 6: Mixed Precision, GPU Memory and Gradient Accumulation

A U-Net has 25,000,000 parameters. Assume (decimal MB, $1\text{ MB} = 10^6\text{ bytes}$):
- Weights, gradients and the two AdamW moment buffers are stored in FP32 (4 bytes each), even under AMP.
- Activations take 150 MB per sample in FP32 and exactly half of that under FP16 autocast.
- Ignore workspace, the autocast weight cache and framework overhead.

### Part A
(a) How many MB do weights + gradients + AdamW state take?
(b) What is the total memory at batch size 16 in FP32 and with AMP?
(c) With a 3,400 MB budget, what is the largest batch size that fits in FP32 and with AMP?

### Part B
You train on the 960-image training split with a micro-batch of 8 and 4 accumulation steps. The last partial batch is not dropped (check that it does not occur here).
(d) What is the effective batch size?
(e) How many micro-batches and how many optimizer steps happen per epoch?
(f) By what factor should each micro-batch loss be divided before `backward()`?
(g) How many warmup steps does a 2-epoch warmup (counted in optimizer steps) take?

**Answer format:** `a (MB); b_fp32, b_amp (MB); c_fp32, c_amp (integers); d; e_micro-batches, e_steps; f; g (integers)`.


In [ ]:
# ==============================================================================
# Question 6: Mixed Precision, GPU Memory and Gradient Accumulation
# ==============================================================================
n_params = 25_000_000
bytes_per_fp32 = 4

# Part A
# (a) Static memory: Weights (1) + Gradients (1) + AdamW 1st & 2nd moments (2) = 4 buffers
static_bytes = n_params * bytes_per_fp32 * 4
ans_a = int(static_bytes / 1e6)  # 400 MB

# (b) Total memory at batch size 16 (FP32: 150 MB/sample, AMP: 75 MB/sample)
act_fp32_per_sample = 150
act_amp_per_sample = 75
b_size = 16

ans_b_fp32 = ans_a + (b_size * act_fp32_per_sample)
ans_b_amp = ans_a + (b_size * act_amp_per_sample)

# (c) Largest batch size fitting within 3,400 MB budget
budget = 3400
act_budget = budget - ans_a  # 3000 MB available for activations
ans_c_fp32 = int(act_budget // act_fp32_per_sample)  # 3000 // 150 = 20
ans_c_amp = int(act_budget // act_amp_per_sample)    # 3000 // 75 = 40

# Part B
n_train = 960
micro_batch = 8
acc_steps = 4

# (d) Effective batch size
ans_d = micro_batch * acc_steps  # 32

# (e) Micro-batches and optimizer steps per epoch
ans_e_micro = n_train // micro_batch  # 120
ans_e_steps = ans_e_micro // acc_steps  # 30

# (f) Factor to divide loss by before backward()
ans_f = acc_steps  # 4

# (g) Warmup steps for a 2-epoch warmup (counted in optimizer steps)
ans_g = 2 * ans_e_steps  # 60

print(f"(a) Static Memory: {ans_a} MB")
print(f"(b) Total Memory at B=16: FP32={ans_b_fp32} MB, AMP={ans_b_amp} MB")
print(f"(c) Max Batch at 3,400 MB: FP32={ans_c_fp32}, AMP={ans_c_amp}")
print(f"(d) Effective batch size: {ans_d}")
print(f"(e) Per epoch: {ans_e_micro} micro-batches, {ans_e_steps} optimizer steps")
print(f"(f) Loss division factor: {ans_f}")
print(f"(g) 2-epoch warmup steps: {ans_g}")

# Final Formatted Answer
q6_answer = f"{ans_a}; {ans_b_fp32}, {ans_b_amp}; {ans_c_fp32}, {ans_c_amp}; {ans_d}; {ans_e_micro}, {ans_e_steps}; {ans_f}; {ans_g}"
print("\n" + "="*40)
print(f"Question 6 Final Answer: {q6_answer}")
print("="*40)
